In [1]:
import glob
import pandas as pd
import numpy as np

In [2]:
# # fetch option oi for a specific date: 
# exp = date(2026, 5, 6)
# df = client.option_history_open_interest(
#         symbol='SPY',
#         expiration=exp,
#         start_date=exp,
#         end_date=exp
#     )

# # merge greek and oi data
# greeks = pd.read_parquet("../data/raw/greeks/SPY/2026-05-06.parquet")
# oi = df.drop(columns=["symbol", "timestamp", "expiration"])

# merged = greeks.merge(
#     oi,
#     on=["strike", "right"],
#     how="left",
# )
# merged["open_interest"] = merged["open_interest"].fillna(0)

In [5]:
# Load and concat all OI files
oi_files = sorted(glob.glob("../data/raw/oi/SPY/*.parquet"))
oi = pd.concat([pd.read_parquet(f) for f in oi_files], ignore_index=True)
oi = oi.drop(columns=["symbol", "timestamp"])

# If multiple OI snapshots per contract, keep the first (pre-market)
oi = oi.drop_duplicates(subset=["expiration", "strike", "right"], keep="first")

# Align types with processed greeks before merging
greeks = pd.read_parquet("../data/processed/spy_processed.parquet")
# spy_processed already has open_interest merged in by the script; drop it so we can re-derive cleanly
greeks = greeks.drop(columns=["open_interest"], errors="ignore")
oi["expiration"] = pd.to_datetime(oi["expiration"])

merged = greeks.merge(oi, on=["expiration", "strike", "right"], how="left")
merged["open_interest"] = merged["open_interest"].fillna(0)


In [6]:
# Check if the processed data generated in notebook and src are logically the same
a = merged  # from notebook cell
b = pd.read_parquet("../data/processed/spy_processed.parquet")

print(f"Shape:  notebook={a.shape}  script={b.shape}")

only_a = set(a.columns) - set(b.columns)
only_b = set(b.columns) - set(a.columns)
print(f"Only in notebook: {only_a}")
print(f"Only in script:   {only_b}")

sort_keys = ["expiration", "timestamp", "strike", "right"]
common = [c for c in a.columns if c in b.columns]
a_s = a[common].sort_values(sort_keys).reset_index(drop=True)
b_s = b[common].sort_values(sort_keys).reset_index(drop=True)

for col in common:
    if pd.api.types.is_numeric_dtype(a_s[col]):
        if not np.allclose(a_s[col].fillna(0), b_s[col].fillna(0), rtol=1e-5, atol=1e-8):
            diff = (a_s[col] - b_s[col]).abs()
            print(f"MISMATCH {col}: max_diff={diff.max():.6g}, mean_diff={diff.mean():.6g}")
    else:
        mismatches = (a_s[col] != b_s[col]).sum()
        if mismatches:
            print(f"MISMATCH {col}: {mismatches} rows differ")

print("Done")


Shape:  notebook=(15652, 23)  script=(15652, 23)
Only in notebook: set()
Only in script:   set()
Done
